# NBA Experiments

Run offline experiments against `NBA Golden Dataset` using the evaluators from `nba_evaluators.ipynb`. Each `evaluate()` call becomes one experiment in the LangSmith UI you can compare side-by-side.

The target function **replays each scripted conversation turn-by-turn** against the compiled graph — one `run_turn(...)` call per user turn, sharing the same `thread_id` so the LangGraph checkpointer accumulates state exactly like a live chat. The last turn's state is what evaluators grade.

Requires `NBA_MOCK=0` (or unset) and the CAI Inference Service LLM endpoint reachable. The risk guardrail is currently the in-process rules-based check in `app/nba_app.py` (`_rules_based_risk_score`), so no classifier endpoint is required for these experiments.

## Setup

In [ ]:
import os
os.environ.setdefault('LANGSMITH_TRACING', 'true')
os.environ.setdefault('LANGSMITH_PROJECT', 'nba-demo')

from dotenv import load_dotenv
load_dotenv(override=True)

In [ ]:
import uuid
import sys
from pathlib import Path

# Application code (nba_app.py, db.py, offer_rules.py, pii_datagen.py) now
# lives under ./app.  Put it on sys.path so ``import nba_app`` resolves.
sys.path.insert(0, str(Path.cwd() / 'app'))

from langsmith import Client

client = Client()
DATASET_NAME = 'NBA Golden Dataset'

## Import the graph + evaluators

We import `run_turn` from `nba_app.py`. Editing prompts / temperatures below relies on reloading the module — restart the kernel between experiments if you edit the app source.

In [ ]:
# Import the app entry point
import nba_app
from nba_app import run_turn

In [ ]:
# --- Evaluators (mirror of nba_evaluators.ipynb) ---
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI

judge_llm = ChatOpenAI(
    model=os.environ.get('LLM_MODEL_ID', 'nemotron'),
    base_url=os.environ.get('LLM_ENDPOINT_BASE_URL'),
    api_key=os.environ.get('LLM_CDP_TOKEN'),
    temperature=0.0,
)

class Relevance(BaseModel):
    score: int = Field(ge=1, le=10)
    reasoning: str

class ConvoQuality(BaseModel):
    score: int = Field(ge=1, le=10)
    reasoning: str

def correct_offer_selection(inputs, outputs, reference_outputs):
    expected_id = reference_outputs.get('expected_offer_id')
    expected_path = reference_outputs.get('expected_path', 'proceed')
    selected = outputs.get('selected_offer') or {}
    selected_id = selected.get('offer_id') if isinstance(selected, dict) else None
    if expected_path == 'decline':
        score = int(selected_id is None and outputs.get('risk_blocked') is True)
        return {'key': 'correct_offer', 'score': score}
    return {'key': 'correct_offer', 'score': int(selected_id == expected_id),
            'comment': f'expected={expected_id}, got={selected_id}'}

def guardrail_correctness(inputs, outputs, reference_outputs):
    expected_decline = reference_outputs.get('expected_path') == 'decline'
    got_decline = bool(outputs.get('risk_blocked'))
    return {'key': 'guardrail_correct', 'score': int(expected_decline == got_decline)}

def offer_relevance_llm_judge(inputs, outputs, reference_outputs):
    if reference_outputs.get('expected_path') == 'decline':
        return {'key': 'offer_relevance', 'score': None, 'comment': 'skipped (decline)'}
    offer = outputs.get('selected_offer') or {}
    if not offer:
        return {'key': 'offer_relevance', 'score': 0, 'comment': 'no offer presented'}
    turns = '\n'.join(f'- {t}' for t in inputs.get('turns', []))
    prompt = f'''Grade this credit-card recommendation 1-10.

Customer said:
{turns}

Bot recommended:
- {offer.get("offer_name")} ({offer.get("offer_id")})
- hook: {offer.get("marketing_hook")}
- target: risk_tier={offer.get("target_risk_tier")}, min_income={offer.get("min_income")}

Be strict. 1=irrelevant, 10=perfect fit.'''
    r = judge_llm.with_structured_output(Relevance).invoke(prompt)
    return {'key': 'offer_relevance', 'score': r.score, 'comment': r.reasoning}

def _format_transcript(messages):
    lines = []
    for m in messages or []:
        role = 'H' if getattr(m, 'type', None) == 'human' or (isinstance(m, dict) and m.get('type') == 'human') else 'A'
        content = getattr(m, 'content', None) if not isinstance(m, dict) else m.get('content', '')
        lines.append(f'{role}: {content}')
    return '\n'.join(lines)

def conversation_quality_llm_judge(inputs, outputs, reference_outputs):
    transcript = _format_transcript(outputs.get('messages', []))
    prompt = f'''Grade this chatbot conversation 1-10 on clarifying-question quality.

Transcript:
{transcript}

Bot took {outputs.get("turn_count_to_offer", -1)} turns to present an offer (expected between {reference_outputs.get("min_turns_to_offer", 2)} and {reference_outputs.get("max_turns_to_offer", 4)}).
Consider: relevant questions? No repetition? Not over-asking? Helpful tone?'''
    r = judge_llm.with_structured_output(ConvoQuality).invoke(prompt)
    return {'key': 'conversation_quality', 'score': r.score, 'comment': r.reasoning}

print('Evaluators loaded.')

## Replay target function

For each dataset example, we open a fresh `thread_id` and invoke `run_turn` once per user turn. LangGraph's checkpointer keeps state across turns exactly as it would in the live app.

In [ ]:
def _count_turns_until_offer(messages) -> int:
    """Turns of *user* input taken before the first `[OFFER PRESENTED: ...]` marker."""
    user_turns = 0
    for m in messages or []:
        mtype = getattr(m, 'type', None) if not isinstance(m, dict) else m.get('type')
        content = getattr(m, 'content', '') if not isinstance(m, dict) else m.get('content', '')
        if mtype == 'human':
            user_turns += 1
        elif isinstance(content, str) and content.startswith('[OFFER PRESENTED:'):
            return user_turns
    return -1  # no offer presented


def _serialize_messages(messages):
    out = []
    for m in messages or []:
        if isinstance(m, dict):
            out.append({'type': m.get('type'), 'content': m.get('content', '')})
        else:
            out.append({'type': getattr(m, 'type', 'ai'), 'content': getattr(m, 'content', '')})
    return out


def target_function(inputs: dict) -> dict:
    """Replay a scripted conversation and return the final state summary."""
    thread_id = f'eval-{uuid.uuid4()}'
    customer_id = inputs.get('customer_id')
    result = None
    for turn in inputs.get('turns', []):
        result = run_turn(turn, thread_id, customer_id)
        # If the risk guardrail already fired, no point continuing the script.
        if result.get('risk_blocked'):
            break
    if result is None:
        return {'selected_offer': None, 'risk_blocked': False, 'turn_count_to_offer': -1, 'messages': []}
    return {
        'selected_offer': result.get('selected_offer'),
        'risk_blocked': result.get('risk_blocked', False),
        'turn_count_to_offer': _count_turns_until_offer(result.get('messages', [])),
        'messages': _serialize_messages(result.get('messages', [])),
    }

In [ ]:
# Quick smoke test on one example
sample = next(iter(client.list_examples(dataset_name=DATASET_NAME)))
print('inputs:', sample.inputs)
out = target_function(sample.inputs)
print('selected_offer:', (out['selected_offer'] or {}).get('offer_id'))
print('risk_blocked:', out['risk_blocked'])
print('turns_to_offer:', out['turn_count_to_offer'])

## Experiment 1 — baseline (temperature 0.2)

Establish the reference metrics.

In [ ]:
EVALUATORS = [
    correct_offer_selection,
    guardrail_correctness,
    offer_relevance_llm_judge,
    conversation_quality_llm_judge,
]

baseline_results = client.evaluate(
    target_function,
    data=DATASET_NAME,
    evaluators=EVALUATORS,
    experiment_prefix='nba-baseline-t0.2',
    metadata={'temperature': 0.2, 'prompt_version': 'v1'},
    max_concurrency=4,
    num_repetitions=1,
)
baseline_results

## Experiment 2 — higher temperature (0.7)

Tests robustness of routing / offer selection under a less deterministic LLM.

Because `nba_app.py` reads the LLM temperature at import time, we monkey-patch the module's LLM instance for this experiment. Restart the kernel to reset.

In [ ]:
# Swap the app's LLM to a higher-temperature clone.
hi_temp_llm = ChatOpenAI(
    model=os.environ.get('LLM_MODEL_ID', 'nemotron'),
    base_url=os.environ.get('LLM_ENDPOINT_BASE_URL'),
    api_key=os.environ.get('LLM_CDP_TOKEN'),
    temperature=0.7,
)
if hasattr(nba_app, 'llm'):
    nba_app.llm = hi_temp_llm  # feature-extraction / clarify / pitch prompts pick this up

hi_temp_results = client.evaluate(
    target_function,
    data=DATASET_NAME,
    evaluators=EVALUATORS,
    experiment_prefix='nba-hi-temp-0.7',
    metadata={'temperature': 0.7, 'prompt_version': 'v1'},
    max_concurrency=4,
    num_repetitions=3,  # stochastic — average over runs
)
hi_temp_results

## Experiment 3 — split-scoped runs

Target only the risk-decline split to isolate guardrail regressions, and the student split to sanity-check the lower-income offer path.

In [ ]:
risk_only = client.evaluate(
    target_function,
    data=client.list_examples(dataset_name=DATASET_NAME, splits=['risk-decline']),
    evaluators=[correct_offer_selection, guardrail_correctness],
    experiment_prefix='nba-risk-split',
    metadata={'split': 'risk-decline'},
    max_concurrency=4,
)
risk_only

In [ ]:
student_only = client.evaluate(
    target_function,
    data=client.list_examples(dataset_name=DATASET_NAME, splits=['student']),
    evaluators=EVALUATORS,
    experiment_prefix='nba-student-split',
    metadata={'split': 'student'},
    max_concurrency=4,
)
student_only

## Acceptance criteria

On the golden dataset with temperature 0.2, we expect:

- `correct_offer >= 0.7`
- `guardrail_correct == 1.0`
- `offer_relevance` mean `>= 7/10`
- `conversation_quality` mean `>= 6/10`

If you fall short, iterate on:
- `FEATURE_EXTRACTION_SYS` prompt in `nba_app.py` (helps with turn merging)
- `INTENT_ROUTER_SYS` prompt (helps clarify vs. present timing)
- Rule-engine weights in `offer_rules.py` (helps offer targeting)

Then re-run Experiment 1 with `experiment_prefix='nba-baseline-t0.2-vN'` to compare.

## Advanced pattern (not implemented) — user simulator

A second LLM plays the customer, driven by a persona in the dataset input; runs until the bot presents an offer or hits a turn cap. Better realism, more compute. Sketch:
```python
def target_with_simulator(inputs):
    thread_id = f'sim-{uuid.uuid4()}'
    persona = inputs['persona']
    convo = []
    for _ in range(6):  # turn cap
        user_msg = simulator_llm.invoke(persona_prompt(persona, convo))
        result = run_turn(user_msg.content, thread_id, inputs.get('customer_id'))
        convo.append({'user': user_msg.content, 'bot': result['assistant_message']})
        if result.get('selected_offer') or result.get('risk_blocked'):
            break
    return {...}
```
Keep scripted-replay as the default; enable the simulator on a subset when adversarial testing.